[代码参考transformer库](https://transformers.run/c2/2021-12-17-transformers-note-4/)

[数据集来源：蚂蚁金融语义相似度数据集AFQMC](https://storage.googleapis.com/cluebenchmark/tasks/afqmc_public.zip)



# 构造Dataset

读取json文件，就是把json文件里面的每一行都读取出来，然后去掉首位的字符

## 映射形Dataset-数据集比较小

继承自 Dataset 类，表示一个从索引到样本的映射（索引可以不是整数），这样我们就可以方便地通过 dataset[idx] 来访问指定索引的样本。这也是目前最常见的数据集类型。

**映射型数据集必须实现 __getitem__() 函数，其负责根据指定的 key 返回对应的样本。一般还会实现 __len__() 用于返回数据集的大小。**

In [17]:
from torch.utils.data import Dataset, DataLoader
import json

class MyDataset(Dataset):
    def __init__(self, data_file_path):
        self.data = self.load_data(data_file_path)
        
    def load_data(self, data_file_path):
        Data = {}
        with open(data_file_path, "rt") as f:
            for idx, line in enumerate(f):                    
                data = json.loads(line.strip())
                Data[idx] = data
        return Data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [18]:
train_data = MyDataset("./afqmc_public/train.json")
valid_data = MyDataset("./afqmc_public/dev.json")

In [19]:
train_data[0]

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}

## 迭代形Dataset-数据集比较大

继承自 IterableDataset，表示可迭代的数据集，它可以通过 iter(dataset) 以数据流 (steam) 的形式访问，适用于访问超大数据集或者远程服务器产生的数据。 

**迭代型数据集必须实现 __iter__() 函数，用于返回一个样本迭代器 (iterator)。**

如果在 DataLoader 中开启多进程（num_workers > 0），那么在加载迭代型数据集时必须进行专门的设置worker_init_fn函数，否则会重复访问样本。

In [20]:
from torch.utils.data import IterableDataset
from torch.utils.data import get_worker_info
import math


import json

class MyDataset(IterableDataset):
    def __init__(self, data_file_path):
        self.data_file_path = data_file_path

    def __iter__(self):
        with open(self.data_file_path, "rt") as f:
            for line in f:
                data = json.loads(line.strip())
                yield data
                
    

In [21]:
train_data = MyDataset("./afqmc_public/train.json")
valid_data = MyDataset("./afqmc_public/dev.json")

In [22]:
next(iter(train_data))

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}

# 构造DataLoader

In [23]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

checkpoint = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def collate_fn(batch_samples):
    batch_sequence_1, batch_sequence_2 = [], []
    batch_label = []
    
    for sample in batch_samples:
        batch_sequence_1.append(sample['sentence1'])
        batch_sequence_2.append(sample['sentence2'])
        batch_label.append(int(sample['label']))
        
    x = tokenizer(
        batch_sequence_1,
        batch_sequence_2,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    
    x["label"] = torch.tensor(batch_label)
    
    return x

# 当指定多个num_loader的时候就需要使用到这个东西
def worker_init_fn(worker_id):
    worker_info = get_worker_info()
    dataset_info = worker_info.dataset
    per_worker_tasks = int(math.floor(dataset_info.end - dataset_info.start) // worker_info.num_workers)
    worker_id = worker_info.id
    dataset_info.start = dataset_info.start + worker_id * per_worker_tasks
    dataset_info.end = min(dataset_info.start + per_worker_tasks, dataset_info.end)

In [24]:
train_loader = DataLoader(dataset=train_data,
                          batch_size=4, collate_fn=collate_fn)

valid_loader = DataLoader(dataset=valid_data, batch_size=4, collate_fn=collate_fn)
for k, v in next(iter(train_loader)).items():
    print(k, v.shape)

input_ids torch.Size([4, 30])
token_type_ids torch.Size([4, 30])
attention_mask torch.Size([4, 30])
label torch.Size([4])


# 构建模型 

In [25]:
from torch import nn
from transformers import AutoConfig
from transformers import BertPreTrainedModel, BertModel

device = "cuda:0" if torch.cuda.is_available() else "cpu"

class BertForPairwiseCLS(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config) # 把一些固定的transformer操作初始化， 比如from_pretrain函数
        self.bert = BertModel(config, add_pooling_layer=False) # 把bert主体填充进去，后面用from_pretrained函数加载参数
        self.dropout = nn.Dropout(config.hidden_dropout_prob) # 自定义的层，pre_trainded无法加载参数，可以用随机的
        self.classifier = nn.Linear(768, 2) # 把Bert最后隐藏层转化成2类
        self.post_init()
        
    def forward(self, x):
        bert_output = self.bert(
            input_ids = x["input_ids"],
            token_type_ids = x["token_type_ids"],
            attention_mask = x["attention_mask"]
        )
        cls_vectors = bert_output.last_hidden_state[:, 0, :] # 每个sample都取第一个token的信息作为评判依据
        logits = self.classifier(cls_vectors)
        return logits
    
config = AutoConfig.from_pretrained(checkpoint)
model = BertForPairwiseCLS.from_pretrained(checkpoint, config=config).to(device) # 会拿checkpoint中的参数放入到BertModel里面，然后其他没有的网络（classifier，dropout），他都会跳过去
print(model)

Some weights of BertForPairwiseCLS were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForPairwiseCLS(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [26]:
outputs = model(next(iter(train_loader)).to(device))
outputs.shape


torch.Size([4, 2])

# 设置训练Loop

这里复现的是单词训练的Loop

In [27]:
from tqdm.auto import tqdm

In [28]:
def train_loop(dataloader, model, loss_fn, optimizer, lr_scheduler, epoch, total_loss, finished_step_num=0):
    progressbar = tqdm(dataloader)
    progressbar.set_description(f"loss:{0:>7f}")
    
    model.train()
    for x in progressbar:
        finished_step_num += 1
        x = x.to(device)
        pred = model(x)
        loss = loss_fn(pred, x["label"])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        total_loss += loss.item()
        progressbar.set_description(f"epoch : {epoch} train_steps : {finished_step_num} loss : {total_loss / (finished_step_num):>7f}")
        progressbar.update(1)
        
    return total_loss, finished_step_num 

# 设置测试Loop

这里复现的是单词测试的Loop

In [35]:
def test_loop(dataloader, model, mode='Test'):
    size = 0
    correct = 0
    
    model.eval()
    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)
            pred = model(x)
            correct += (pred.argmax(dim=-1) == x["label"]).sum().item()
            size += pred.shape[0]
            print(f"correct : {correct} size{size}")
    
    correct /= size
    print(f"{mode} Accuracy: {(100*correct):>0.1f}%\n")
    

# 添加其他配件

In [30]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [31]:
from transformers import get_scheduler

epoches = 3
num_training_steps = epoches * len(list(train_data))

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

num_training_steps


103002

# 正式开始训练

In [36]:
from transformers import  get_scheduler
from torch.optim import AdamW

learning_rate = 1e-5
epoche_num = 3

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = epoche_num * len(list(train_data))
)

total_loss = 0
finished_step_num = 0

print(f"Device : {device} \n")
for ep in range(epoche_num):
    print(f"Epoch {ep}\n-------------------------------")
    
    total_loss, finished_step_num = train_loop(train_loader, model, loss_fn, optimizer, lr_scheduler, ep, total_loss, finished_step_num)
    test_loop(valid_loader, model, "Valid")
print("Done")  

Device : cuda:0 

Epoch 0
-------------------------------


epoch : 0 train_steps : 8584 loss : 0.168042: : 8584it [10:24, 13.75it/s]


correct : 2 size4
correct : 5 size8
correct : 7 size12
correct : 9 size16
correct : 13 size20
correct : 16 size24
correct : 20 size28
correct : 24 size32
correct : 26 size36
correct : 27 size40
correct : 30 size44
correct : 33 size48
correct : 36 size52
correct : 39 size56
correct : 42 size60
correct : 44 size64
correct : 48 size68
correct : 51 size72
correct : 55 size76
correct : 58 size80
correct : 60 size84
correct : 61 size88
correct : 64 size92
correct : 67 size96
correct : 70 size100
correct : 74 size104
correct : 78 size108
correct : 80 size112
correct : 83 size116
correct : 85 size120
correct : 89 size124
correct : 93 size128
correct : 95 size132
correct : 99 size136
correct : 101 size140
correct : 102 size144
correct : 106 size148
correct : 109 size152
correct : 112 size156
correct : 116 size160
correct : 117 size164
correct : 119 size168
correct : 122 size172
correct : 125 size176
correct : 129 size180
correct : 132 size184
correct : 135 size188
correct : 138 size192
correct 

epoch : 1 train_steps : 17168 loss : 0.152039: : 8584it [11:29, 12.45it/s]


correct : 2 size4
correct : 5 size8
correct : 9 size12
correct : 11 size16
correct : 15 size20
correct : 17 size24
correct : 19 size28
correct : 23 size32
correct : 26 size36
correct : 28 size40
correct : 29 size44
correct : 31 size48
correct : 34 size52
correct : 37 size56
correct : 39 size60
correct : 42 size64
correct : 46 size68
correct : 49 size72
correct : 53 size76
correct : 56 size80
correct : 58 size84
correct : 59 size88
correct : 62 size92
correct : 65 size96
correct : 67 size100
correct : 71 size104
correct : 73 size108
correct : 75 size112
correct : 79 size116
correct : 81 size120
correct : 85 size124
correct : 87 size128
correct : 90 size132
correct : 92 size136
correct : 95 size140
correct : 96 size144
correct : 99 size148
correct : 102 size152
correct : 106 size156
correct : 110 size160
correct : 112 size164
correct : 114 size168
correct : 117 size172
correct : 120 size176
correct : 124 size180
correct : 127 size184
correct : 130 size188
correct : 134 size192
correct : 

epoch : 2 train_steps : 25752 loss : 0.137473: : 8584it [12:04, 11.84it/s]


correct : 2 size4
correct : 5 size8
correct : 9 size12
correct : 11 size16
correct : 15 size20
correct : 17 size24
correct : 19 size28
correct : 22 size32
correct : 24 size36
correct : 26 size40
correct : 27 size44
correct : 30 size48
correct : 33 size52
correct : 36 size56
correct : 39 size60
correct : 41 size64
correct : 45 size68
correct : 48 size72
correct : 52 size76
correct : 55 size80
correct : 58 size84
correct : 60 size88
correct : 63 size92
correct : 65 size96
correct : 67 size100
correct : 71 size104
correct : 75 size108
correct : 77 size112
correct : 80 size116
correct : 82 size120
correct : 86 size124
correct : 89 size128
correct : 92 size132
correct : 95 size136
correct : 98 size140
correct : 99 size144
correct : 103 size148
correct : 106 size152
correct : 109 size156
correct : 113 size160
correct : 114 size164
correct : 115 size168
correct : 118 size172
correct : 121 size176
correct : 125 size180
correct : 128 size184
correct : 129 size188
correct : 132 size192
correct :